# Практическая работа 6. Признаки для прогноза повторного заказа

Вариант 8

Наблюдение — один пользователь в момент первого зарегистрированного заказа. Цель — узнать, оформит ли он ещё один заказ в ближайшие 24 часа. Заказом считаем факт оформления при любом последующем статусе.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from shop_data import DATA, read_events, read_orders

orders = read_orders().sort_values(["order_datetime", "order_id"])
events = read_events()
products = pd.read_excel(DATA / "products.xlsx", sheet_name="products")
print("Число заказов на пользователя за квартал:")
print(orders.groupby("user_id")["order_id"].nunique().agg(["min", "max"]))

Число заказов на пользователя за квартал:
min    122
max    177
Name: order_id, dtype: int64


За весь квартал каждый пользователь оформил от 122 до 177 заказов. Признак «больше одного заказа за квартал» был бы равен единице у всех. Окно в 24 часа задаёт одинаковый срок наблюдения после первого заказа и позволяет выделить оба класса.

### 1. Что известно при первом заказе

Берём сумму, количество товаров, час оформления и категорию первого заказа. Из логов добавляем число событий, зарегистрированных к этому моменту. Статусы заказов и последующую активность в признаки не включаем.

In [2]:
first = orders.drop_duplicates("user_id").copy()
first = first.rename(columns={"order_datetime": "first_time"})
first = first.merge(products[["product_id", "category_name"]], on="product_id", validate="many_to_one")
first = first.set_index("user_id")

history = events.join(first["first_time"], on="user_id", how="inner")
before = history[history["event_time"] <= history["first_time"]]
features = pd.DataFrame({
    "first_amount": (first["price"] * first["quantity"]).round(2),
    "first_quantity": first["quantity"],
    "first_hour": first["first_time"].dt.hour,
    "events_before_order": before.groupby("user_id").size().reindex(first.index, fill_value=0),
})
# Категории заранее известны из справочника товаров.
categories = pd.Categorical(first["category_name"], categories=sorted(products["category_name"].unique()))
category_flags = pd.get_dummies(pd.Series(categories, index=first.index), prefix="category", dtype=int)
features = features.join(category_flags)
features.head()

,first_amount,first_quantity,first_hour,events_before_order,category_Electronics,category_Fashion,category_Home,category_Sport,category_Toys
user_id,,,,,,,,,
1028,3638.42,1,0,0,0,0,0,0,1
1090,37393.83,3,0,0,0,0,0,1,0
1004,37935.33,3,0,0,0,0,1,0,0
1007,58653.03,3,0,0,0,0,0,0,1
1031,3759.60,3,0,0,0,1,0,0,0


### 2. Целевая переменная

Включаем первый заказ и считаем различные `order_id` за следующие сутки. Если окно выходит за конец наблюдения, пользователя исключаем.

In [3]:
day = pd.Timedelta(days=1)
complete = first.index[first["first_time"] + day <= orders["order_datetime"].max()]
followup = orders.join(first["first_time"], on="user_id")
in_window = followup["order_datetime"].between(followup["first_time"], followup["first_time"] + day)
orders_count = followup.loc[in_window].groupby("user_id")["order_id"].nunique()
dataset = features.loc[complete].copy()
dataset["is_repeated_buyer"] = orders_count.loc[complete].gt(1).astype(int)
dataset["is_repeated_buyer"].value_counts().sort_index()

is_repeated_buyer
0    21
1    79
Name: count, dtype: int64

### 3. Обучающая и тестовая выборки

Разделяем пользователей в пропорции 80/20 с сохранением соотношения классов. Идентификатор остаётся индексом и в CSV не записывается.

In [4]:
train, test = train_test_split(
    dataset, test_size=0.2, random_state=8, stratify=dataset["is_repeated_buyer"]
)
out = Path("results")
out.mkdir(exist_ok=True)
train.to_csv(out / "train.csv", index=False)
test.to_csv(out / "test.csv", index=False)

pd.DataFrame({
    "train": train["is_repeated_buyer"].value_counts(),
    "test": test["is_repeated_buyer"].value_counts(),
}).sort_index()

,train,test
is_repeated_buyer,,
0,17,4
1,63,16


Повторный заказ в течение суток оформили 79 пользователей из 100. В выборках 80 и 20 покупателей, каждый представлен одной строкой.